# Data Cleaning Process

## Loading the data

### Import the Libraries

In [7]:
import pandas as pd
import numpy as np

### Load the dataset

In [8]:
dataset = pd.read_csv("C:/Users/mille/Desktop/python_projects/cafe_cleaning_data/data/01_raw/cafe_sales.csv")
dataset

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


## Exploring the data

In [9]:
#Make a copy of the dataset, so we don't work over the original data
df = dataset.copy()

#Review 10 random rows of data
df.sample(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
7653,TXN_9662632,Salad,4,5.0,20.0,Digital Wallet,In-store,2023-08-10
4662,TXN_6637989,Smoothie,1,4.0,4.0,Cash,NaN,2023-11-09
68,TXN_8427104,Salad,2,ERROR,10.0,NaN,In-store,2023-10-27
5094,TXN_1093668,Coffee,2,2.0,4.0,Digital Wallet,ERROR,2023-05-18
2700,TXN_7602467,Cookie,1,1.0,1.0,Credit Card,Takeaway,2023-10-17
1112,TXN_9725369,Coffee,UNKNOWN,2.0,6.0,NaN,Takeaway,2023-02-10
5310,TXN_6000890,Coffee,3,2.0,6.0,Digital Wallet,In-store,2023-01-06
277,TXN_2130245,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-11-13
1445,TXN_2494138,Cookie,3,1.0,3.0,Cash,NaN,2023-01-10
8595,TXN_7468084,Coffee,5,2.0,10.0,Credit Card,NaN,2023-06-14


In [10]:
#A look to the statistics of the dataset
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


We can already see that it has some null values and errors in the data

In [11]:
#Check the structure of the data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


This helps us identify that all dtypes are object, which is wrong for most of them.

In [12]:
#Count how many null values are per column
total_null_values = df.isnull().sum()
nulls_perct = df.isnull().mean() * 100

print(f'Total Nulls: \n{total_null_values}\n')
print(f'Nulls Percent: \n{nulls_perct}')

Total Nulls: 
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Nulls Percent: 
Transaction ID       0.00
Item                 3.33
Quantity             1.38
Price Per Unit       1.79
Total Spent          1.73
Payment Method      25.79
Location            32.65
Transaction Date     1.59
dtype: float64


The number of null values may seem a problem for payment methods and locations, we need to find out why there are so many.

In [13]:
#We identified that many columns contain the words unknown and error (pending to spot more)
print('Columns with words "Unkown" or "Error"\n')
for i in df.columns:
    print(f'{i}: {df[i].str.contains("unknown|error", case=False, na=False).sum()}')
    

Columns with words "Unkown" or "Error"

Transaction ID: 0
Item: 636
Quantity: 341
Price Per Unit: 354
Total Spent: 329
Payment Method: 599
Location: 696
Transaction Date: 301


## Fixing the data

### Fixing entries that show errors

In [14]:
#First we verify that the words Unknown and Error are unique categories and not part of the dataset
for col in ["Item", "Payment Method", "Location"]:
    print(df[col].value_counts(dropna=False), "\n")

Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64 

Payment Method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64 

Location
NaN         3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64 



In this case we can see that those two words are the only miss-processed data. We decided to replace them all with null values so we can fix the data types of all columns

In [15]:
#Replace all "Unknown" and "Error" words with null values

# r: makes it a raw string, which tells Python to treat backslashes as plain characters instead of escape codes.
# (?i): ignore capitalization
# ^ and $: the match must span the entire cell, from start to end
# \s*: allow any number of spaces before and after
# (unknown|error): either word

df = df.replace(
    to_replace=r"(?i)^\s*(unknown|error)\s*$",
    value=np.nan,
    regex=True
)

In [16]:
#We verify one more time that the changes were applied.
for col in ["Item", "Payment Method", "Location"]:
    print(df[col].value_counts(dropna=False), "\n")

Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
NaN          969
Name: count, dtype: int64 

Payment Method
NaN               3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64 

Location
NaN         3961
Takeaway    3022
In-store    3017
Name: count, dtype: int64 



### Fixing the structure

In [17]:
# Clean column names
df.columns = (df.columns
                .str.strip() #Removes any accidental spaces at the very beginning or end of the column names
                .str.lower() #Converts all characters to lowercase
                .str.replace(" ", "_") #Swaps any spaces between words with underscores
                .str.replace(r"[^\w]", "", regex=True)) #Uses a regular expression to find and delete any special characters (like @, #, %, or punctuation), leaving behind only letters, numbers, and underscores.

df.head()



,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [18]:
#Fix string data types
str_cols = ['item','payment_method','location']
df[str_cols] = df[str_cols].apply(lambda x: x.str.strip().str.lower())

#Fix numeric data types
num_cols = ['quantity','price_per_unit','total_spent']
df[num_cols] = (df[num_cols]
                .replace({r'\$': '', ',': ''}, regex=True)
                .apply(pd.to_numeric, errors='coerce'))

df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  object        
 1   item              9031 non-null   object        
 2   quantity          9521 non-null   float64       
 3   price_per_unit    9467 non-null   float64       
 4   total_spent       9498 non-null   float64       
 5   payment_method    6822 non-null   object        
 6   location          6039 non-null   object        
 7   transaction_date  9540 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 625.1+ KB


In [19]:
df.sample(5)

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
4911,TXN_9030656,juice,5.0,3.0,15.0,digital wallet,takeaway,2023-12-12
5565,TXN_6158518,cookie,5.0,1.0,5.0,cash,NaN,2023-09-28
6348,TXN_5844851,smoothie,5.0,4.0,20.0,NaN,NaN,2023-02-15
8904,TXN_2386202,salad,4.0,5.0,20.0,cash,in-store,2023-01-22
7920,TXN_6070787,sandwich,3.0,4.0,12.0,NaN,in-store,2023-02-26


In [20]:
#Confirm the null values percentage

round(df.isnull().sum() / len(df) * 100,2)

transaction_id       0.00
item                 9.69
quantity             4.79
price_per_unit       5.33
total_spent          5.02
payment_method      31.78
location            39.61
transaction_date     4.60
dtype: float64

We can see the columns for Payment Method and Location are the highest ones. However as we don't have a way to know what's the correct input, we'll replace NAN values with "not_specified"

In [21]:
df["payment_method"] = df["payment_method"].fillna("not_specified")
df["location"] = df["location"].fillna("not_specified")
df["item"] = df["item"].fillna("not_specified")

print(df["payment_method"].unique())
print(df["location"].unique())
print(df["item"].unique())

['credit card' 'cash' 'not_specified' 'digital wallet']
['takeaway' 'in-store' 'not_specified']
['coffee' 'cake' 'cookie' 'salad' 'smoothie' 'not_specified' 'sandwich'
 'juice' 'tea']


We have columns such as quantity, price_per_unit and total_spent that we could try and complete the missing values by making the mathematical calculations needed.

In [22]:
condition = df['total_spent'].isnull() & df['quantity'].notnull() & df['price_per_unit'].notnull()
total_matches = condition.sum()

print(f"Missing values total: {df['total_spent'].isnull().sum()}")

print(f"Total spent nulls with valid quantity and price per unit: {total_matches}")

Missing values total: 502
Total spent nulls with valid quantity and price per unit: 462


Now that we know we can fill most of the values for total_spent, we can apply some basic math to complete it.

In [23]:
df["total_spent"] = df["total_spent"].fillna(df["price_per_unit"] * df["quantity"])

df.isnull().sum()

transaction_id        0
item                  0
quantity            479
price_per_unit      533
total_spent          40
payment_method        0
location              0
transaction_date    460
dtype: int64

Now that the nulls dropped to 40 we can try filling out the quantity and the price_per_unit missing

In [24]:
condition2 = df['total_spent'].notnull() & df['quantity'].notnull() & df['price_per_unit'].isnull()
total_matches = condition2.sum()

print(f"Missing values total: {df['price_per_unit'].isnull().sum()}")

print(f"price_per_unit nulls with others not null: {total_matches}")

Missing values total: 533
price_per_unit nulls with others not null: 495


In [25]:

df["price_per_unit"] = df["price_per_unit"].fillna(df["total_spent"] / df["quantity"])

df.isnull().sum()

transaction_id        0
item                  0
quantity            479
price_per_unit       38
total_spent          40
payment_method        0
location              0
transaction_date    460
dtype: int64

Finally we do the same for quantity

In [26]:
condition3 = df['total_spent'].notnull() & df['quantity'].isnull() & df['price_per_unit'].notnull()
total_matches = condition3.sum()

print(f"Missing values total: {df['quantity'].isnull().sum()}")

print(f"quantity nulls with others not null: {total_matches}")

Missing values total: 479
quantity nulls with others not null: 441


In [27]:
df["quantity"] = df["quantity"].fillna(df["total_spent"] / df["price_per_unit"])

df.isnull().sum()

transaction_id        0
item                  0
quantity             38
price_per_unit       38
total_spent          40
payment_method        0
location              0
transaction_date    460
dtype: int64